# Family D — Router-Training **Probe** (Kaggle 2×T4, 4-bit)  — reviewed v2

LR / load-balance probe for `report/router_training_plan.md`. The full,
confound-free Family-D run is on the RunPod H100 in BF16 — **not here**.

**v2 fixes (3-pass review):** aux loss now computed *after* the forward from
detached gate inputs so gradient-checkpointing can't detach it; `torch_dtype=
float16` set (T4 has no bf16 → avoids dtype clashes); multi-GPU aux terms moved
to a common device; model-parallel flags + single grad-checkpoint enable;
all-masked-label guard.

**Caveats (why this is a probe):** base is 4-bit/fp16 (Families A–C are BF16) →
quantization confound; only the *chosen LR* carries to the real run. T4 is slow →
`MAX_STEPS≈100`.

**Setup:** Accelerator = **GPU T4 ×2**, Internet = **On**. Add the base model and
`WONDERLAND_FINAL_MASTER.jsonl` (as a dataset) to inputs; fix the two paths below.
Note: training needs **no** capture-script cache patches (those were generation-only;
here `use_cache=False`).

In [ ]:
# 1. Deps (Internet ON). Use the mamba-ssm / causal-conv1d versions that worked
#    when you trained the LoRAs on Kaggle — the source build can take ~15-20 min
#    and must match the installed torch; prefer a prebuilt-wheel dataset if you have one.
!pip -q install -U bitsandbytes accelerate peft "transformers==4.57.3" einops
!pip -q install causal-conv1d --no-build-isolation
!pip -q install mamba-ssm --no-build-isolation
import torch; print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 2. CONFIG — edit the two paths; set LR per probe run.
MODEL_PATH  = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
DATA_PATH   = "/kaggle/input/wonderland/WONDERLAND_FINAL_MASTER.jsonl"
OUT_DIR     = "/kaggle/working/familyD_probe"
LR          = 1e-4     # run once each: 1e-4, 3e-5, 1e-5
MAX_STEPS   = 100
AUX_COEF    = 0.01     # load-balance alpha (NOT LoRA alpha)
BALANCE_CAP = 0.08     # abort if any expert > 8% of tokens
MAX_LEN     = 1536
PER_DEV_BATCH, GRAD_ACCUM, SEED = 1, 8, 42
MOE_LAYERS = [1,3,6,8,10,13,15,17,20,22,24,27,29,31,34,36,38,40,43,45,47,49,51]
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 3. Load tokenizer + 4-bit base (fp16 compute) sharded across both T4s.
import torch, torch.nn.functional as F
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          TrainingArguments, Trainer, TrainerCallback)
from peft import prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, quantization_config=bnb, device_map="auto",
    torch_dtype=torch.float16,                 # T4 has no bf16 -> keep non-quant parts fp16
    trust_remote_code=True, attn_implementation="eager")
# enables input-grads + gradient checkpointing (needed: base frozen, only gate trains)
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False
model.is_parallelizable = True; model.model_parallel = True   # device_map -> no DataParallel
print("device map (sample):", dict(list(model.hf_device_map.items())[:4]))

In [ ]:
# 4. Freeze all, unfreeze ONLY the 23 gate.weight (cast fp32). Sanity-assert.
for p in model.parameters(): p.requires_grad_(False)
n_train = 0
for L in MOE_LAYERS:
    w = model.backbone.layers[L].mixer.gate.weight
    w.data = w.data.float(); w.requires_grad_(True); n_train += w.numel()
GATES = {L: model.backbone.layers[L].mixer.gate for L in MOE_LAYERS}
assert n_train > 0 and all(g.weight.requires_grad for g in GATES.values())
assert sum(p.requires_grad for p in model.parameters()) == len(MOE_LAYERS)
print(f"trainable router params: {n_train:,} across {len(GATES)} gates (rest frozen)")

In [ ]:
# 5. Load-balance aux loss — computed AFTER the forward from detached gate inputs,
#    so gradient checkpointing can't detach the term; grad still reaches gate.weight.
#    aux = N * sum_e f_e * P_e   (P_e diff'able via gate.weight; f_e detached counts)
class BalanceHooks:
    def __init__(self, gates):
        self.store, self.handles = {}, []
        for L, g in gates.items():
            self.handles.append(g.register_forward_hook(self._mk(L)))
    def _mk(self, L):
        def hook(m, inp, out): self.store[L] = inp[0].detach().reshape(-1, inp[0].shape[-1])
        return hook
    def clear(self): self.store = {}
    def compute(self, gates, dev):
        terms, max_load = [], 0.0
        for L, h in self.store.items():
            g = gates[L]; h = h.to(g.weight.device)
            scores = F.linear(h.float(), g.weight.float()).sigmoid()
            N = g.n_routed_experts; P_e = scores.mean(0)
            with torch.no_grad():
                idx = g.get_topk_indices(scores)
                oh = torch.zeros(scores.shape[0], N, device=scores.device); oh.scatter_(1, idx, 1.0)
                f_e = oh.mean(0); max_load = max(max_load, float(f_e.max()))
            terms.append((N * torch.sum(f_e * P_e)).to(dev))      # move to common (loss) device
        aux = torch.stack(terms).mean() if terms else torch.zeros((), device=dev)
        return aux, max_load
hooks = BalanceHooks(GATES)

In [ ]:
# 6. Dataset (assistant-only loss masking) + collator, with all-masked guard.
import json
from torch.utils.data import Dataset
class WonderlandSFT(Dataset):
    def __init__(self, path, tok, max_len):
        self.rows = [json.loads(l)["messages"] for l in open(path) if l.strip()]
        self.tok, self.max_len = tok, max_len
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        m = self.rows[i]
        full = self.tok.apply_chat_template(m, tokenize=True, add_generation_prompt=False)[:self.max_len]
        prompt = self.tok.apply_chat_template(m[:-1], tokenize=True, add_generation_prompt=True)
        lab = list(full)
        for j in range(min(len(prompt), len(lab))): lab[j] = -100
        if lab and all(x == -100 for x in lab): lab[-1] = full[-1]   # guard vs NaN loss
        return {"input_ids": full, "labels": lab}
class PadCollator:
    def __init__(self, pad): self.pad = pad
    def __call__(self, batch):
        Lm = max(len(b["input_ids"]) for b in batch); ids, lab, att = [], [], []
        for b in batch:
            n = Lm - len(b["input_ids"])
            ids.append(b["input_ids"] + [self.pad]*n); lab.append(b["labels"] + [-100]*n)
            att.append([1]*len(b["input_ids"]) + [0]*n)
        return {"input_ids": torch.tensor(ids), "labels": torch.tensor(lab),
                "attention_mask": torch.tensor(att)}
ds = WonderlandSFT(DATA_PATH, tok, MAX_LEN); print("examples:", len(ds))

In [ ]:
# 7. Trainer (aux loss + collapse monitor). grad-checkpointing already enabled in cell 3.
HIST = []
class RouterTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        hooks.clear()
        out = model(**inputs); lm = out.loss
        aux, ml = hooks.compute(GATES, lm.device)
        loss = lm + AUX_COEF * aux
        self._last = (float(lm.detach()), float(aux.detach()), ml)
        return (loss, out) if return_outputs else loss
class BalanceMonitor(TrainerCallback):
    def __init__(self): self.bad = 0
    def on_log(self, args, state, control, **kw):
        if hasattr(trainer, "_last"):
            lm, aux, ml = trainer._last; HIST.append((state.global_step, lm, aux, ml))
            print(f"  step {state.global_step}: lm={lm:.4f} aux={aux:.4f} max_load={ml:.3f}", flush=True)
            self.bad = self.bad + 1 if ml > BALANCE_CAP else 0
            if self.bad >= 3:
                print(f"  !! COLLAPSE max_load>{BALANCE_CAP} — stopping (lower LR / add bias-refresh)")
                control.should_training_stop = True
targs = TrainingArguments(
    output_dir=OUT_DIR, per_device_train_batch_size=PER_DEV_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, max_steps=MAX_STEPS,
    lr_scheduler_type="cosine", warmup_ratio=0.05, optim="adamw_torch",
    weight_decay=0.0, max_grad_norm=1.0, fp16=True, logging_steps=5,
    save_strategy="no", seed=SEED, gradient_checkpointing=False,  # already on from cell 3
    report_to="none", remove_unused_columns=False)
trainer = RouterTrainer(model=model, args=targs, train_dataset=ds,
                        data_collator=PadCollator(tok.pad_token_id))
trainer.add_callback(BalanceMonitor())
trainer.train()

In [ ]:
# 8. Save trained gate weights + plot balance/loss trajectory.
gate_sd = {f"backbone.layers.{L}.mixer.gate.weight": GATES[L].weight.detach().cpu() for L in MOE_LAYERS}
torch.save(gate_sd, f"{OUT_DIR}/router_state_probe_lr{LR}.pt")
import matplotlib.pyplot as plt
if HIST:
    s = [h[0] for h in HIST]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(s, [h[1] for h in HIST]); ax[0].set_title(f"LM loss (LR={LR})"); ax[0].set_xlabel("step")
    ax[1].plot(s, [h[3] for h in HIST]); ax[1].axhline(BALANCE_CAP, ls='--', c='r')
    ax[1].set_title("max expert load (red = collapse cap)"); ax[1].set_xlabel("step")
    plt.tight_layout(); plt.show()
print("saved", f"{OUT_DIR}/router_state_probe_lr{LR}.pt")

## Reading the probe
- **LM loss falls** → router is learning. **max-load stays under the red cap** → LR is safe.
- Run **1e-4 / 3e-5 / 1e-5**; pick the **highest LR that stays balanced**.
- Hand that LR to the RunPod full run (BF16, confound-free):
```
python scripts/train_router.py --model-path nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 \
   --data WONDERLAND_FINAL_MASTER.jsonl --out-dir /workspace/ws2/familyD --lr <chosen> --epochs 1
```
Probe weights are throwaway (quantized-trained); only the LR carries over.